# BenoStreamDB: 15-Minute Interactive Tour (Part 2)
## Production RAG Pipeline on SciFact Literature with OpenCode Go

This notebook demonstrates an end-to-end **Retrieval-Augmented Generation (RAG)** pipeline using:
1. **BenoStreamDB** as the high-throughput vector and metadata retrieval engine.
2. The **SciFact** peer-reviewed scientific literature corpus (`mteb/scifact`) from Hugging Face.
3. **OpenCode Go** (`deepseek-v4.1-flash`) for objective claim verification with document-level citations.

### 1. Imports and Client Configuration

In [1]:
import os
import shutil
import warnings
import pandas as pd
import numpy as np
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from openai import OpenAI
import benostreamdb as bsdb

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 90)
pd.set_option('display.width', 1000)

# Configure OpenCode Go Client
OPENCODE_API_KEY = os.environ.get("OPENCODE_API_KEY")
llm_client = OpenAI(
    base_url="https://opencode.ai/zen/go/v1",
    api_key=OPENCODE_API_KEY or "dummy",
    default_headers={"x-opencode-session": "benolabs-howto"}
)
print("OpenCode Go client configured.")

/home/ralbright/projects/benolabs/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


OpenCode Go client configured.


### 2. Download and Embed SciFact Scientific Literature

Download 250 peer-reviewed scientific abstracts from Hugging Face and generate dense 384-dimensional embeddings using `sentence-transformers/all-MiniLM-L6-v2`.

In [2]:
print("Loading SciFact dataset from Hugging Face...")
try:
    corpus = load_dataset("mteb/scifact", "corpus", split="corpus[:250]")
except Exception:
    corpus = load_dataset("BeIR/scifact", "corpus", split="corpus[:250]")

df = pd.DataFrame(corpus).rename(columns={"_id": "doc_id"})

print(f"Generating embeddings for {len(df)} documents...")
model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(df["text"].tolist(), show_progress_bar=False)
df["embedding"] = [list(e) for e in embeddings]
df[["doc_id", "title"]].head()

Loading SciFact dataset from Hugging Face...


Generating embeddings for 250 documents...


Loading weights:   0%|                                 | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|█████████████████████| 103/103 [00:00<00:00, 9342.85it/s]

,doc_id,title
0,4983,Microstructural development of human newborn cerebral white matter assessed in vivo by...
1,5836,Induction of myelodysplasia by myeloid-derived suppressor cells.
2,7912,"BC1 RNA, the transcript from a master gene for ID element amplification, is able to pr..."
3,18670,The DNA Methylome of Human Peripheral Blood Mononuclear Cells
4,19238,The human myelin basic protein gene is included within a 179-kilobase transcription un...


### 3. Ingest into BenoStreamDB Table

Store in a clean local table directory and attach HNSW vector and inverted scalar indexes.

In [3]:
db_path = "scifact_rag_db"
if os.path.exists(db_path):
    shutil.rmtree(db_path)

table = bsdb.Table(db_path, explain=True)
table.add_index_columns(["embedding", "doc_id", "title"])
table.write_pandas(df)
table.commit()
print("Table successfully written and indexed.")

Table successfully written and indexed.


### 4. Vector Retrieval Function

In [4]:
def retrieve(query: str, k: int = 3, max_distance: float = 1.3):
    """Retrieve most relevant scientific papers from BenoStreamDB."""
    query_vec = list(model.encode(query))
    res = table.filter(vector_filter=query_vec, k=k).to_pandas()
    dist_col = "distance" if "distance" in res.columns else "_distance"
    if max_distance is not None and not res.empty:
        res = res[res[dist_col] <= max_distance]
    return res

# Test retrieval
sample_claim = "Microstructural development of human cerebral cortex and white matter can be measured in vivo."
retrieved = retrieve(sample_claim, k=3)
dist_col = "distance" if "distance" in retrieved.columns else "_distance"
retrieved[["doc_id", "title", dist_col]]

------------------------------------------------------------
BENOSTREAM QUERY PLAN [Table: file:///home/ralbright/projects/benolabs/benostreamdb/examples/notebooks/scifact_rag_db]
Context Selection:
  -> Total Table Scope: 250 rows in 1 segments
  -> Execution Scope: 250 rows in 1 segments

Vector Execution:
  -> VectorSearch (col: embedding, k: 3, metric: L2)
     [Access: Brute Force Scan (No Index)] [Eligibility: 250 rows]

Final Retrieval:
  -> ParallelRead (threads: 16, format: Parquet)
------------------------------------------------------------


,doc_id,title,distance
0,4983,Microstructural development of human newborn cerebral white matter assessed in vivo by...,0.581540
1,803312,Cerebral organoids model human brain development and microcephaly,1.131012


### 5. Synthesize Answer with OpenCode Go (Disabled Reasoning Traces)

Pass retrieved scientific contexts to `deepseek-v4.1-flash` via OpenCode Go with `{"thinking": {"type": "disabled"}}`.

In [5]:
def evaluate_claim(claim: str, k: int = 3) -> str:
    retrieved_df = retrieve(claim, k=k)
    if retrieved_df.empty:
        return "No sufficiently relevant literature found in the database."

    dist_col = "distance" if "distance" in retrieved_df.columns else "_distance"
    context_blocks = []
    for _, row in retrieved_df.iterrows():
        context_blocks.append(
            f"[Doc ID: {row['doc_id']}] Title: {row['title']}\n"
            f"Abstract: {row['text'][:400]}... (Distance: {row[dist_col]:.4f})"
        )
    combined_context = "\n\n".join(context_blocks)

    prompt = f"""You are an expert scientific research assistant. Evaluate the following scientific claim based ONLY on the provided peer-reviewed literature context.
Cite specific document IDs in your answer.

Claim: {claim}

Retrieved Literature:
{combined_context}

Provide a concise, factual evaluation in 2-3 sentences."""

    if OPENCODE_API_KEY:
        try:
            response = llm_client.chat.completions.create(
                model="deepseek-v4.1-flash",
                messages=[
                    {"role": "system", "content": "You are an objective scientific evaluation assistant. Answer concisely and cite document IDs."},
                    {"role": "user", "content": prompt}
                ],
                max_tokens=250,
                temperature=0.2,
                extra_body={"thinking": {"type": "disabled"}}
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            print(f"[Warning: OpenCode Go call failed ({e}); falling back to local citation]")

    top = retrieved_df.iloc[0]
    return f"[Synthesized from Doc {top['doc_id']}: {top['title']}] Evidence indicates: {top['text'][:200]}..."""

### 6. End-to-End Scientific Claim Evaluations

In [6]:
claim1 = "Microstructural development of human cerebral cortex and white matter can be measured in vivo."
print(f"Claim: {claim1}\n")
print(evaluate_claim(claim1))

Claim: Microstructural development of human cerebral cortex and white matter can be measured in vivo.

------------------------------------------------------------
BENOSTREAM QUERY PLAN [Table: file:///home/ralbright/projects/benolabs/benostreamdb/examples/notebooks/scifact_rag_db]
Context Selection:
  -> Total Table Scope: 250 rows in 1 segments
  -> Execution Scope: 250 rows in 1 segments

Vector Execution:
  -> VectorSearch (col: embedding, k: 3, metric: L2)
     [Access: Brute Force Scan (No Index)] [Eligibility: 250 rows]

Final Retrieval:
  -> ParallelRead (threads: 16, format: Parquet)
------------------------------------------------------------


The claim is supported by Doc ID 4983, which applied diffusion tensor MRI in vivo to measure microstructural properties (apparent diffusion coefficient, relative anisotropy, fiber architecture) of human newborn cerebral white matter. However, the retrieved context addresses white matter microstructure specifically; direct in vivo measurement of cortical microstructure is not clearly documented in these abstracts. Doc ID 803312 concerns an in vitro cerebral organoid model and does not support the in vivo claim.


In [7]:
claim2 = "Cerebral organoids model human brain development and microcephaly pathogenesis."
print(f"Claim: {claim2}\n")
print(evaluate_claim(claim2))

Claim: Cerebral organoids model human brain development and microcephaly pathogenesis.

------------------------------------------------------------
BENOSTREAM QUERY PLAN [Table: file:///home/ralbright/projects/benolabs/benostreamdb/examples/notebooks/scifact_rag_db]
Context Selection:
  -> Total Table Scope: 250 rows in 1 segments
  -> Execution Scope: 250 rows in 1 segments

Vector Execution:
  -> VectorSearch (col: embedding, k: 3, metric: L2)
     [Access: HNSW-IVF Cluster Index] [Eligibility: 250 rows]

Final Retrieval:
  -> ParallelRead (threads: 16, format: Parquet)
------------------------------------------------------------


The claim is supported by Doc ID 803312, which reports a human pluripotent stem cell-derived three-dimensional cerebral organoid culture system that develops discrete, interdependent brain regions, directly modeling human brain development and microcephaly. The other retrieved documents (Doc IDs 4983 and 980008) address white matter microstructure and MeCP2 overexpression in mice, respectively, and do not bear on this claim.


### 7. Clean Teardown

In [8]:
if os.path.exists(db_path):
    shutil.rmtree(db_path)
print("Cleaned up RAG database.")

Cleaned up RAG database.
